# Week 2 — Full-network confirmation (Colab)

Runs the **full** male-CNS network (176,422 neurons, 24.3M connections) at the chosen `w_syn` and at 1.0×, to check what the subnetworks can't:
1. Does MN9 L still respond to sugar once all the surrounding inhibition is present?
2. Do the descending candidates still respond to JO-C/E?
3. Does the whole network stay quiet (no runaway)?

**Runtime → Change runtime type → CPU** (Brian2 doesn't use the GPU). Use **High-RAM** if it's offered. Results save to Drive after each condition, so a disconnect loses at most one run.

### Setup: Drive and Brian2

On a fresh runtime this installs Brian2 and restarts the runtime (Colab shows a crash notice; that's expected). Then choose **Runtime → Run all** again.

In [ ]:
import sys, subprocess
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    try:
        import brian2
    except ImportError:
        # installing brian2 can upgrade numpy, which breaks the numpy already loaded:
        # install, then restart the runtime. Afterwards, run all cells again.
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'brian2'], check=True)
        print("brian2 installed -> restarting runtime. When it reconnects, choose Runtime -> Run all again.")
        import os; os.kill(os.getpid(), 9)
except ImportError:
    IN_COLAB = False

from pathlib import Path
BRAIN = Path('/content/drive/MyDrive/arm_simulation/brain') if IN_COLAB else Path.home() / 'arm_simulation/brain'
DATA = BRAIN / 'data'
sys.path.insert(0, str(BRAIN))
for p in ['sim/lif.py', 'data/weights_male-cns_v1.0/meta.json', 'data/shortlist_week1.csv', 'data/wsyn_choice.json']:
    print(f"{p:45s} {'ok' if (BRAIN / p).exists() else 'MISSING'}")

### Load weights and groups (same definitions as `03_simulation.ipynb`)

In [ ]:
import json, time
import numpy as np, pandas as pd
from sim.lif import WeightSet, LIFNet, peak_mem_gb
from brian2 import Hz, second
from brian2.codegen.runtime.cython_rt import CythonCodeObject

ws = WeightSet(DATA / 'weights_male-cns_v1.0')
CHOSEN = json.loads((DATA / 'wsyn_choice.json').read_text())['w_scale']

JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
SUGAR = ws.by_type(r'^LB3(b|c)$', cls='cb_sensory')
MN9 = ws.by_type(r'^MN9$')
shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
LEG_MN = g2i.loc[shortlist.query("group == 'output_T1_leg_MN'").bodyId].to_numpy()

print(f"chosen w_scale: {CHOSEN}")
print(f"JO-C/E {len(JO)}, sugar {len(SUGAR)}, MN9 {len(MN9)}, DN candidates {len(DN_cand)}, leg MNs {len(LEG_MN)}")
print(f"Cython available: {CythonCodeObject.is_available()}, peak memory {peak_mem_gb():.2f} GB")

### Build the full network

Built once, with Poisson inputs on both sugar GRNs and JO-C/E. Each run switches on only one group (the other gets rate 0). Expect a few minutes and several GB of memory.

In [ ]:
sub = ws.subnetwork(np.arange(ws.N))
STIM = np.concatenate([SUGAR, JO])
net = LIFNet(sub, stim_global=STIM)
n_conn = len(sub['w'])
del sub

def rates_for(group, hz=150):
    return np.where(np.isin(STIM, group), hz, 0) * Hz

### Run conditions

Timing is printed after the first run. If one 1 s trial takes more than ~10 minutes, lower `N_TRIALS` to 1.

In [ ]:
N_TRIALS = 2
OUT = DATA / 'fullnet_week2.csv'
done = pd.read_csv(OUT) if OUT.exists() else pd.DataFrame()

conditions = [('baseline', None, CHOSEN), ('sugar', SUGAR, CHOSEN), ('JO-C/E', JO, CHOSEN),
              ('sugar', SUGAR, 1.0), ('JO-C/E', JO, 1.0)]
for name, group, sc in conditions:
    if len(done) and ((done.condition == name) & (done.w_scale == sc)).any():
        print(f"skip {name} @ {sc}x (already saved)"); continue
    t0 = time.time()
    rate = 0 * Hz if group is None else rates_for(group)
    r = net.rates(net.run(rate=rate, n_trials=N_TRIALS, w_scale=sc, verbose=False), ws)
    others = r.drop(STIM)
    row = dict(condition=name, w_scale=sc, trials=N_TRIALS,
               MN9_L=r.rate_hz[MN9[0]], MN9_R=r.rate_hz[MN9[1]],
               DN_cand_firing=int((r.loc[DN_cand, 'rate_hz'] >= 1).sum()),
               DN_cand_mean_hz=r.loc[DN_cand, 'rate_hz'].mean(),
               legMN_firing=int((r.loc[LEG_MN, 'rate_hz'] >= 1).sum()),
               legMN_mean_hz=r.loc[LEG_MN, 'rate_hz'].mean(),
               network_firing=(others.rate_hz >= 1).mean(),
               network_firing_n=int((others.rate_hz >= 1).sum()),
               minutes=round((time.time() - t0) / 60, 1))
    done = pd.concat([done, pd.DataFrame([row])], ignore_index=True)
    done.to_csv(OUT, index=False)
    r.to_parquet(DATA / f"fullnet_rates_{name.replace('/', '')}_{sc}x.parquet")
    print(f"{name:8s} @ {sc}x: {row['minutes']} min | MN9 L {row['MN9_L']:.1f} Hz | "
          f"DN cand {row['DN_cand_mean_hz']:.1f} Hz | leg MN {row['legMN_firing']} | "
          f"network firing {row['network_firing']:.2%} | peak memory {peak_mem_gb():.1f} GB")

display(done.round(3))

### Which classes fire?

Shows where activity spreads in the full network at the chosen scale. A large share of optic lobe or unrelated classes firing would mean runaway.

In [ ]:
for name in ['sugar', 'JO-C/E']:
    f = DATA / f"fullnet_rates_{name.replace('/', '')}_{CHOSEN}x.parquet"
    if f.exists():
        r = pd.read_parquet(f).drop(STIM)
        t = r.groupby('cls').rate_hz.agg(neurons='size', firing=lambda x: int((x >= 1).sum()))
        t['share'] = (t.firing / t.neurons).round(3)
        print(f"{name} @ {CHOSEN}x")
        display(t.sort_values('firing', ascending=False).head(12))